# Final Matching and Ranking

This notebook combines the results from the previous notebooks.

The final matching system uses:

- TF-IDF similarity
- Word2Vec similarity
- Skill coverage

The previous notebooks already calculated these features, so they are loaded and combined here instead of being calculated again.

The final score is then used to rank candidates for job positions.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
MODELS = PROJECT_DIR / "Models"
DATA = PROJECT_DIR / "Data"
MODELS.mkdir(parents=True, exist_ok=True)


## 1. Load the Existing Results

The TF-IDF and Word2Vec results were already generated in the previous notebooks.

We will load these existing results and use them as inputs for the final matching system.

In [10]:
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name == "Python Files":
    PROJECT_DIR = PROJECT_DIR.parent

DATA = PROJECT_DIR / "Data"
MODELS = PROJECT_DIR / "Models"

print("Project:", PROJECT_DIR)
print("Data:", DATA)

Project: D:\Instant AI\AI\Graduation Project\My Project
Data: D:\Instant AI\AI\Graduation Project\My Project\Data


In [11]:
tfidf_df = pd.read_csv(DATA / "tfidf_baseline_results.csv")

word2vec_df = pd.read_csv(DATA / "word2vec_results.csv")

print("TF-IDF shape:", tfidf_df.shape)
print("Word2Vec shape:", word2vec_df.shape)


TF-IDF shape: (9544, 36)
Word2Vec shape: (9544, 34)


## 2. Check the Available Columns

Before combining the results, we check the column names in both files.

This helps us identify the similarity columns and avoid creating duplicate or incorrect columns.

In [12]:
print("TF-IDF columns:")
print(tfidf_df.columns.tolist())

print()

print("Word2Vec columns:")
print(word2vec_df.columns.tolist())

TF-IDF columns:
['career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'skills_required', 'matched_score', 'candidate_text', 'job_text', 'candidate_text_length', 'job_text_length', 'split', 'tfidf_similarity', 'absolute_error']

Word2Vec columns:
['career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'start_dates', 'end_dates', 'related_skils

## 3. Combine TF-IDF and Word2Vec Results

The two result files contain the same 9,544 candidate-job pairs.

TF-IDF provides the `tfidf_similarity` value, while Word2Vec provides the `word2vec_similarity` value.

We will take these two similarity columns and add them to one final DataFrame.

In [13]:
skill_df = pd.read_csv(DATA / "skills_matching_results.csv")
assert len(tfidf_df) == len(word2vec_df) == len(skill_df)
assert (tfidf_df["job_position_name"] == word2vec_df["job_position_name"]).all()
assert (tfidf_df["job_position_name"] == skill_df["job_position_name"]).all()


In [14]:
df = tfidf_df.copy()

df["word2vec_similarity"] = word2vec_df["word2vec_similarity"]

print("Final DataFrame shape:", df.shape)

Final DataFrame shape: (9544, 37)


In [15]:
print(
    df[
        [
            "job_position_name",
            "tfidf_similarity",
            "word2vec_similarity"
        ]
    ].head(10)
)

                                   job_position_name  tfidf_similarity  \
0                           senior software engineer          0.003533   
1                     machine learning (ml) engineer          0.056484   
2  executive/ senior executive- trade marketing, ...          0.000000   
3                     business development executive          0.012055   
4                                senior ios engineer          0.004766   
5                                        ai engineer          0.014965   
6                                senior ios engineer          0.002348   
7                                senior ios engineer          0.006297   
8                                mechanical engineer          0.002118   
9                     business development executive          0.001965   

   word2vec_similarity  
0             0.692982  
1             0.834400  
2             0.473196  
3             0.681219  
4             0.549506  
5             0.717050  
6         

## 4. Load Skill Matching Results

The skill matching and skill coverage calculations were completed in Notebook 06.

The results from Notebook 06 will be loaded and combined with the TF-IDF and Word2Vec results.

In [16]:
skill_df = pd.read_csv(DATA / "skills_matching_results.csv")

print("Skill matching shape:", skill_df.shape)


Skill matching shape: (9544, 47)


In [17]:
print(skill_df.columns.tolist())

['career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'skills_required', 'matched_score', 'candidate_text', 'job_text', 'candidate_text_length', 'job_text_length', 'candidate_skills', 'required_skills', 'candidate_skills_normalized', 'required_skills_normalized', 'candidate_skills_final', 'candidate_skills_extracted', 'required_skills_extracted', 'required_skills_final', 'required_skill_count', 'skill_coverage', 'matched_skills', 'missing_skills', 'matched_skill_count', 'ski

## 5. Check the Skill Matching Results

The skill matching results should contain the skill coverage calculated in Notebook 06.

We check the available columns before selecting the required result.

In [18]:
print(
    skill_df[
        [
            "job_position_name",
            "skill_coverage_percentage"
        ]
    ].head(10)
)

                                   job_position_name  \
0                           senior software engineer   
1                     machine learning (ml) engineer   
2  executive/ senior executive- trade marketing, ...   
3                     business development executive   
4                                senior ios engineer   
5                                        ai engineer   
6                                senior ios engineer   
7                                senior ios engineer   
8                                mechanical engineer   
9                     business development executive   

   skill_coverage_percentage  
0                   0.000000  
1                  37.878788  
2                   0.000000  
3                  20.000000  
4                   0.000000  
5                   3.571429  
6                   5.555556  
7                   5.555556  
8                   0.000000  
9                  10.000000  


## 6. Add Skill Coverage to the Final DataFrame

The skill coverage percentage has already been calculated in Notebook 06.

We will add this existing result to the current DataFrame without recalculating the skill matching.

In [19]:
df["skill_coverage_percentage"] = skill_df["skill_coverage_percentage"]
df["skill_coverage_score"] = df["skill_coverage_percentage"] / 100   # stays NaN where no required skills

df["matched_skills"] = skill_df["matched_skills"]
df["missing_skills"] = skill_df["missing_skills"]

df["candidate_record_id"] = np.arange(1, len(df) + 1)

print("Skill coverage added:", df["skill_coverage_percentage"].notna().sum())
print("Matched skills column added:", "matched_skills" in df.columns)
print("Missing skills column added:", "missing_skills" in df.columns)

Skill coverage added: 9544
Matched skills column added: True
Missing skills column added: True


In [20]:
df[
    [
        "job_position_name",
        "tfidf_similarity",
        "word2vec_similarity",
        "skill_coverage_percentage"
    ]
].head(20)

,job_position_name,tfidf_similarity,word2vec_similarity,skill_coverage_percentage
0,senior software engineer,0.003533,0.692982,0.000000
1,machine learning (ml) engineer,0.056484,0.834400,37.878788
2,"executive/ senior executive- trade marketing, ...",0.000000,0.473196,0.000000
3,business development executive,0.012055,0.681219,20.000000
4,senior ios engineer,0.004766,0.549506,0.000000
5,ai engineer,0.014965,0.717050,3.571429
6,senior ios engineer,0.002348,0.506417,5.555556
7,senior ios engineer,0.006297,0.577048,5.555556
8,mechanical engineer,0.002118,0.557009,0.000000
9,business development executive,0.001965,0.574242,10.000000


## 7. Convert Skill Coverage to a Score

Skill coverage is stored as a percentage from 0 to 100.

For combining it with TF-IDF and Word2Vec similarities, it is converted to a value between 0 and 1.

## 8. Calculate the Final Match Score

The final match score combines three existing signals:

- TF-IDF similarity
- Word2Vec similarity
- Skill coverage

The TF-IDF and Word2Vec scaling values are learned from the training portion only. The saved scaling parameters are then applied to both training and test rows. This prevents the evaluation portion from influencing the score normalization.

The final weights remain the project design choice: 25% TF-IDF, 25% Word2Vec, and 50% skill coverage.


In [21]:
from project_utils import scale_value, combine_scores

if "split" not in df.columns:
    raise ValueError("The TF-IDF results must contain the train/test split created in Notebook 03.")

train_mask = df["split"].eq("train")
test_mask = df["split"].eq("test")

if train_mask.sum() == 0 or test_mask.sum() == 0:
    raise ValueError("Both train and test rows are required before final scoring.")

train_df = df.loc[train_mask].copy()

scaling = {}
for col in ["tfidf_similarity", "word2vec_similarity"]:
    low, high = train_df[col].quantile(0.05), train_df[col].quantile(0.95)
    scaling[col] = {"low": float(low), "high": float(high)}
    print(col, scaling[col])

joblib.dump(scaling, MODELS / "score_scaling.pkl")

df["tfidf_scaled"] = df["tfidf_similarity"].apply(
    lambda x: scale_value(x, **scaling["tfidf_similarity"])
)
df["word2vec_scaled"] = df["word2vec_similarity"].apply(
    lambda x: scale_value(x, **scaling["word2vec_similarity"])
)

df["final_match_score"] = [
    combine_scores(t, w, s)
    for t, w, s in zip(
        df["tfidf_scaled"],
        df["word2vec_scaled"],
        df["skill_coverage_score"]
    )
]
df["final_match_percentage"] = df["final_match_score"] * 100

print("Training rows:", train_mask.sum())
print("Test rows:", test_mask.sum())


tfidf_similarity {'low': 0.0, 'high': 0.04455240867866502}
word2vec_similarity {'low': 0.5051747010000001, 'high': 0.784518155}
Training rows: 7635
Test rows: 1909


## 9. Inspect Final Match Score

The final match percentage provides a combined score using the three similarity and skill-based signals.

This score is used as a baseline ranking measure for comparing candidate-job pairs.

In [22]:
df[
    [
        "job_position_name",
        "final_match_percentage"
    ]
].describe()

,final_match_percentage
count,9544.000000
mean,25.412741
std,17.960671
min,0.000000
25%,11.514444
50%,21.951302
75%,35.947826
max,100.000000


In [23]:
df[
    [
        "job_position_name",
        "tfidf_similarity",
        "word2vec_similarity",
        "skill_coverage_score",
        "final_match_percentage"
    ]
].sort_values(
    "final_match_percentage",
    ascending=False
).head(20)

,job_position_name,tfidf_similarity,word2vec_similarity,skill_coverage_score,final_match_percentage
8680,executive/ sr. executive -it,0.074758,0.816854,1.000000,100.000000
1559,mechanical engineer,0.070694,0.786367,0.900000,95.000000
5694,mechanical engineer,0.100783,0.793787,0.900000,95.000000
8809,mechanical engineer,0.046054,0.764948,0.900000,93.248591
5857,mechanical engineer,0.068182,0.764354,0.900000,93.195375
5112,executive/ sr. executive -it,0.196617,0.834917,0.833333,91.666667
2447,executive/ sr. executive -it,0.062026,0.834690,0.833333,91.666667
3116,senior software engineer,0.046436,0.745629,0.875000,90.269566
6676,mechanical engineer,0.120336,0.784283,0.800000,89.978982
1775,mechanical designer,0.091513,0.822723,0.777778,88.888889


## 10. Analyze the Distribution of Final Match Scores

The distribution helps us understand how the final matching scores are spread across all candidate-job pairs.

## 11. Compare the Matching Components

The following scatter plots are used to examine the relationships between the individual matching signals and the final match score.

## 12. Final Matching Results

This section summarizes the final candidate-job matching results and checks the highest-scoring candidate-job pairs.

In [24]:
print("========== FINAL MATCHING CHECK ==========")

print("Total rows:", len(df))

print(
    "Average final match percentage:",
    df["final_match_percentage"].mean()
)

print(
    "Minimum final match percentage:",
    df["final_match_percentage"].min()
)

print(
    "Maximum final match percentage:",
    df["final_match_percentage"].max()
)

========== FINAL MATCHING CHECK ==========
Total rows: 9544
Average final match percentage: 25.412741086536386
Minimum final match percentage: 0.0
Maximum final match percentage: 100.0


## 13. Final Data Validation

The following checks make sure the final matching dataset contains the expected number of rows, both train and test rows are present, and the final score does not contain invalid values.


In [25]:
print("========== FINAL DATA VALIDATION ==========")

print("Total rows:", len(df))
print("Train rows:", (df["split"] == "train").sum())
print("Test rows:", (df["split"] == "test").sum())
print("Missing final scores:", df["final_match_percentage"].isna().sum())
print("Scores below 0:", (df["final_match_percentage"] < 0).sum())
print("Scores above 100:", (df["final_match_percentage"] > 100).sum())


========== FINAL DATA VALIDATION ==========
Total rows: 9544
Train rows: 7635
Test rows: 1909
Missing final scores: 0
Scores below 0: 0
Scores above 100: 0


In [26]:
df[
    [
        "job_position_name",
        "tfidf_similarity",
        "word2vec_similarity",
        "skill_coverage_score",
        "final_match_percentage"
    ]
].head(20)

,job_position_name,tfidf_similarity,word2vec_similarity,skill_coverage_score,final_match_percentage
0,senior software engineer,0.003533,0.692982,0.000000,18.790379
1,machine learning (ml) engineer,0.056484,0.834400,0.378788,68.939394
2,"executive/ senior executive- trade marketing, ...",0.000000,0.473196,0.000000,0.000000
3,business development executive,0.012055,0.681219,0.200000,32.519589
4,senior ios engineer,0.004766,0.549506,0.000000,6.641692
5,ai engineer,0.014965,0.717050,0.035714,29.144933
6,senior ios engineer,0.002348,0.506417,0.055556,4.206383
7,senior ios engineer,0.006297,0.577048,0.055556,12.743399
8,mechanical engineer,0.002118,0.557009,0.000000,5.827411
9,business development executive,0.001965,0.574242,0.100000,12.283874


## 14. Save Final Matching Dataset

The completed dataset is saved as a CSV file for use in the next stage of the project.

In [27]:
output_path = DATA / "final_matching_results.csv"

df.to_csv(
    output_path,
    index=False
)

print("Final dataset saved successfully.")
print(output_path)


Final dataset saved successfully.
D:\Instant AI\AI\Graduation Project\My Project\Data\final_matching_results.csv


## 15. Final Project Summary

The dataset now contains the main matching signals required for the candidate-job matching system:

- TF-IDF similarity
- Word2Vec similarity
- Skill coverage score
- Final match percentage

These features provide the foundation for evaluating candidate-job compatibility and for the next stage of the project.